In [1]:
import numpy as np
import pandas as pd
import cv2
from matplotlib import pyplot as plt
import os
from os import listdir
from os.path import isfile, isdir, join
from os import walk

from numpy.random import random
from numpy.random import rand

import time

# Load table

In [2]:
table_cap1 = np.load('cap1.npy')
table_cap2 = np.load('cap3_2.npy')
table_cap3 = np.load('cap5.npy')

table_wli1 = np.load('wli1.npy')

# Define Base Function

In [3]:
def transfer_by_table(img, table):
    
    # reshape as nx3, and to int
    temp_img = img.reshape(-1, 3).astype(int)

    # transfer to table key
    img_keys = temp_img[:,0] + np.left_shift(temp_img[:,1], 8) + np.left_shift(temp_img[:,2], 16)

    # table mapping
    img_values = table[img_keys]

    # reconstruct to rgb image
    img_r = img_values.astype('uint8')
    img_g = np.right_shift(img_values, 8).astype('uint8')
    img_b = np.right_shift(img_values, 16).astype('uint8')

    img_trans = np.column_stack((img_r, img_g, img_b))

    img_trans = img_trans.reshape(img.shape)
    
    return img_trans

In [4]:
def filter_ext(files, tar_ext):
    
    tar_files = []
    
    for file in files:
        
        file_ext = os.path.splitext(file)[-1][1:].lower()
        
        if file_ext == tar_ext:
            tar_files.append(file)
        
    return tar_files
    
def get_all_ext_list(tar_path, tar_ext):
    
    all_tar_files = []
    
    for root, dirs, files in walk(tar_path):
        
        tar_files = filter_ext(files, tar_ext.lower())

        if len(tar_files) == 0:
            continue
            
        tar_files = [join(root, file) for file in tar_files]
            
        all_tar_files = all_tar_files + tar_files
        
    return all_tar_files

In [5]:
def to_nbi(wli, is_dual = False, trans_table = None):
    
    h,w,c = wli.shape

    image_rgb = cv2.cvtColor(wli, cv2.COLOR_BGR2RGB)
    
    pred_img2 = None
    
    if trans_table is None:
        pred_img2 = image_rgb.copy()
    else:
        pred_img2 = transfer_by_table(image_rgb, trans_table)

    pred_img2 = cv2.cvtColor(pred_img2, cv2.COLOR_RGB2BGR)
    
    if is_dual:
        tar_img = np.zeros((h, w*2, c))
        tar_img[:h,:w,:] = wli
        tar_img[:h,w:2*w,:] = pred_img2
    else:
        tar_img = np.zeros((h, w, c))
        tar_img[:h,:w,:] = pred_img2
        
    tar_img = tar_img.astype('uint8')
    
    return tar_img

# Define Target Path

In [117]:
#src_path = r'C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule'
#tar_path = r'C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi'

src_path = r'G:\New_datasets_SAVE\Katie\WLI\train\Serrated'
tar_path = r'G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated'

# Get All Files

In [118]:
res_files = get_all_ext_list(src_path, 'png')
#res_files

# Batch Transfer Images

In [119]:
is_dual = False
tar_table = table_wli1

for index, file in enumerate(res_files):
    
    dump_file = file.replace(src_path, tar_path)
    dump_file_path = os.path.dirname(os.path.abspath(dump_file))
    
    if not os.path.isdir(dump_file_path):
        try:
            os.makedirs(dump_file_path)
        except FileExistsError:
            print("建立路徑異常。")
            
    img = cv2.imread(file)
    nbi = to_nbi(img, is_dual=is_dual, trans_table=tar_table)
    
    cv2.imwrite(dump_file, nbi)
    
    print(index, dump_file)

0 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_1.png
1 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_10.png
2 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_10_aug1.png
3 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_10_aug2.png
4 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_10_aug3.png
5 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_10_aug4.png
6 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_10_aug5.png
7 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_10_aug6.png
8 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_11.png
9 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_11_aug1.png
10 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_11_aug2.png
11 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_11_aug3.png
12 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\serrated_03_11_aug4.png
13 G:\New_datasets_SAVE\Katie\SAVE2\train\Serrated\

In [9]:
# 裁切影像

In [17]:
src_path = r'C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus'
tar_path = r'C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut'

In [18]:
res_files = get_all_ext_list(src_path, 'jpg')

In [20]:
for index, file in enumerate(res_files):
    
    dump_file = file.replace(src_path, tar_path)
    dump_file_path = os.path.dirname(os.path.abspath(dump_file))
    
    if not os.path.isdir(dump_file_path):
        try:
            os.makedirs(dump_file_path)
        except FileExistsError:
            print("建立路徑異常。")
            
    img = cv2.imread(file)
    
    # Capsule
#     img = img[22:922, 694:1894,:]

    # Olympus
    if img.shape[1] == 640:
        img = img[24:472, 93:611,:]
    elif img.shape[1] == 1440:
        img = img[42:1036, 280:1422,:]
        
    cv2.imwrite(dump_file, img)
    
    print(index, dump_file)

0 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-1.jpg
1 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-2.jpg
2 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-3.jpg
3 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-4.jpg
4 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-5.jpg
5 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-6.jpg
6 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-7.jpg
7 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-8.jpg
8 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x001\33178799-9.jpg
9 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x002\33582739-1.jpg
10 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x002\33582739-10.jpg
11 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x002\33582739-11.jpg
12 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x

109 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x017\41620524-3.jpg
110 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x017\41620524-5.jpg
111 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x017\41620524-6.jpg
112 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x017\41620524-7.jpg
113 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x017\41620524-8.jpg
114 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x018\41675836-14.jpg
115 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x018\41675836-2.jpg
116 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x018\41675836-3.jpg
117 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x018\41675836-4.jpg
118 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x019\25286355-1.jpg
119 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x019\25286355-2.jpg
120 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x019\25286355-3.jpg
121 C:\Users\Tony\Desktop\TEST\10_capsu

229 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x034\37941518-4.jpg
230 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x034\37941518-6.jpg
231 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x034\37941518-7.jpg
232 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x034\37941518-8.jpg
233 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x034\37941518-9jpg.jpg
234 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x035\26856042-1.jpg
235 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x035\26856042-2.jpg
236 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x036\46812684-1.jpg
237 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x036\46812684-2.jpg
238 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x036\46812684-3.jpg
239 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x036\46812684-4.jpg
240 C:\Users\Tony\Desktop\TEST\10_capsule\image_olympus_cut\x037\40019452-1.jpg
241 C:\Users\Tony\Desktop\TEST\10_cap

In [ ]:
res_files

In [23]:
res_files = get_all_ext_list(r'E:\Capsule\capsule_org\20220530_IRB', 'mp4')

In [39]:
res_files

['E:\\Capsule\\capsule_org\\20220530_IRB\\20201120\\X001\\E000000019\\S000000019\\V000000018.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201124\\X002\\V000000019.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201127\\X003\\E000000021\\S000000021\\V000000020.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201127\\X003\\E000000021\\S000000021\\V000000021.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201127\\X004\\E000000022\\S000000022\\V000000022.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201201\\X006\\E000000023\\S000000023\\V000000023.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201201\\X006\\E000000023\\S000000023\\V000000024.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201203\\X009\\E000000024\\S000000024\\V000000025.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201204\\X005\\E000000027\\S000000027\\V000000029.mp4',
 'E:\\Capsule\\capsule_org\\20220530_IRB\\20201204\\X007\\E000000026\\S000000026\\V000000028.mp4',
 'E:\\Capsule\\capsule_org\\202205

In [38]:
for file in res_files:
#     print(file)
    print(file[45:49]+','+file[-14:-4]) 

X001,V000000018
X002,V000000019
X003,V000000020
X003,V000000021
X004,V000000022
X006,V000000023
X006,V000000024
X009,V000000025
X005,V000000029
X007,V000000028
X011,V000000026
X011,V000000027
X008,V000000030
X012,V000000031
X014,V000000032
test,V000000035
X015,V000000033
X015,V000000034
X017,V000000036
X013,V000000037
X018,V000000038
X019,V000000039
X020,V000000040
X021,V000000041
X022,V000000044
X023,V000000043
X024,V000000042
X025,V000000045
X025,V000000048
X026,V000000046
X028,V000000047
X027,V000000049
X029,V000000050
X030,V000000051
X31\,V000000052
X32\,V000000053
X033,V000000054
X034,V000000055
X035,V000000056
X038,V000000057
X036,V000000059
X037,V000000058
X041,V000000060
X044,V000000061
X042,V000000062
X045,V000000063
X046,V000000068
X047,V000000067
